In [ ]:
!pip install datasets matplotlib seaborn scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from pathlib import Path

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f" Hardware ativo para processamento: {device}")

In [ ]:
def configurar_reprodutibilidade_medica(semente: int = 42):
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(semente)
        torch.cuda.manual_seed_all(semente)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

configurar_reprodutibilidade_medica(42)
print(" Reprodutibilidade científica configurada!")

In [ ]:
print("A carregar o dataset do Hugging Face...")
dataset_hf = load_dataset("BTX24/tekno21-brain-stroke-dataset-binary")

labels_total = dataset_hf['train']['label']
saudaveis = labels_total.count(0)
avc = labels_total.count(1)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, ax = plt.subplots(figsize=(6, 4))
barras = ax.bar(['Saudável (Classe 0)', 'AVC (Classe 1)'], [saudaveis, avc], color=['#2ca02c', '#d62728'], width=0.5, edgecolor='black', linewidth=1.2)

for barra in barras:
    altura = barra.get_height()
    ax.annotate(f'{altura}', xy=(barra.get_x() + barra.get_width() / 2, altura), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.title('Distribuição Total de Pacientes no Dataset', fontsize=13, fontweight='bold', pad=15)
plt.ylabel('Quantidade de Amostras', fontsize=11)
plt.ylim(0, max(saudaveis, avc) * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

dados_divididos = dataset_hf['train'].train_test_split(test_size=0.3, seed=42)
train_hf = dados_divididos['train']
temp_hf = dados_divididos['test']

dados_val_test = temp_hf.train_test_split(test_size=0.5, seed=42)
val_hf = dados_val_test['train']
test_hf = dados_val_test['test']

print(f"\n Divisão Concluída -> Treino: {len(train_hf)} | Validação: {len(val_hf)} | Teste: {len(test_hf)}")

In [ ]:
img_saudavel, img_avc = None, None

for item in dataset_hf['train']:
    if item['label'] == 0 and img_saudavel is None:
        img_saudavel = item['image']
    elif item['label'] == 1 and img_avc is None:
        img_avc = item['image']
    if img_saudavel is not None and img_avc is not None:
        break

fig, eixos = plt.subplots(1, 2, figsize=(9, 4.5))

eixos[0].imshow(img_saudavel, cmap='gray')
eixos[0].set_title("Exemplo: Paciente Saudável", fontsize=12, color='green', fontweight='bold')
eixos[0].axis('off')

eixos[1].imshow(img_avc, cmap='gray')
eixos[1].set_title("Exemplo: Paciente com AVC", fontsize=12, color='red', fontweight='bold')
eixos[1].axis('off')

plt.suptitle("Inspeção de Imagens Médicas (Tomografias)", fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
class StrokeDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.hf_dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        item = self.hf_dataset[idx]
        imagem = item['image'].convert('L')
        rotulo = float(item['label'])

        if self.transform:
            imagem = self.transform(imagem)

        return imagem, torch.tensor([rotulo])

transformacao_treino = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transformacao_teste = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset_treino = StrokeDataset(train_hf, transform=transformacao_treino)
dataset_val = StrokeDataset(val_hf, transform=transformacao_treino)
dataset_teste = StrokeDataset(test_hf, transform=transformacao_treino)

loader_treino = DataLoader(dataset=dataset_treino, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
loader_val = DataLoader(dataset=dataset_val, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
loader_teste = DataLoader(dataset=dataset_teste, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)
print("DataLoaders criados e prontos!")

In [ ]:
class RedeCNN(nn.Module):
    def __init__(self):
        super(RedeCNN, self).__init__()

        self.convolucoes = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classificador = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 14 * 14, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.convolucoes(x)
        x = self.classificador(x)
        return x

modelo = RedeCNN().to(device)
print("Arquitetura profunda com 256 neurónios carregada com sucesso!")

In [ ]:
criterio = nn.BCEWithLogitsLoss()
otimizador = optim.Adam(modelo.parameters(), lr=0.001, weight_decay= 1e-4)

epocas = 30
paciencia = 5
caunt_paciencia = 1

historico_perda_treino = []
historico_perda_validacao = []
historico_acuracia_treino = []
historico_acuracia_validacao = []

melhor_acuracia_valid = 0.0
print("Iniciando o Treinamento...\n")

for epoca in range(epocas):
    modelo.train()
    perda_acumulada = 0.0
    treino_corretos = 0
    treino_total = 0

    for imagens, rotulos in loader_treino:
        imagens, rotulos = imagens.to(device), rotulos.to(device)
        otimizador.zero_grad()
        saidas = modelo(imagens)
        perda = criterio(saidas, rotulos)
        perda.backward()
        otimizador.step()

        perda_acumulada += perda.item()

        previsoes_treino = (torch.sigmoid(saidas) >= 0.5).float()
        treino_total += rotulos.size(0)
        treino_corretos += (previsoes_treino == rotulos).sum().item()

    media_perda_epoca = perda_acumulada / len(loader_treino)
    historico_perda_treino.append(media_perda_epoca)
    acuracia_treino = (treino_corretos / treino_total) * 100
    historico_acuracia_treino.append(acuracia_treino)

    modelo.eval()
    perda_val_acumulada = 0.0
    val_corretos = 0
    val_total = 0

    todos_rotulos = []
    todas_previsoes = []

    with torch.no_grad():
        for imagens, rotulos in loader_val:
            imagens, rotulos = imagens.to(device), rotulos.to(device)
            saidas = modelo(imagens)

            perda_val = criterio(saidas, rotulos)
            perda_val_acumulada += perda_val.item()

            previsoes = (torch.sigmoid(saidas) >= 0.5).float()
            val_total += rotulos.size(0)
            val_corretos += (previsoes == rotulos).sum().item()

            todos_rotulos.extend(rotulos.cpu().numpy())
            todas_previsoes.extend(previsoes.cpu().numpy())

    historico_perda_validacao.append(perda_val_acumulada / len(loader_val))
    acuracia_val = (val_corretos / val_total) * 100
    historico_acuracia_validacao.append(acuracia_val)

    precisao_macro = precision_score(todos_rotulos, todas_previsoes, average='macro', zero_division=0)
    recall_macro = recall_score(todos_rotulos, todas_previsoes, average='macro', zero_division=0)
    f1_macro = f1_score(todos_rotulos, todas_previsoes, average='macro', zero_division=0)

    print(f'Época [{epoca+1:03d}/{epocas}]')
    print(f'  -> Perda Treino: {media_perda_epoca:.4f} | Acurácia Treino: {acuracia_treino:.2f}% | Acurácia Val: {acuracia_val:.2f}%')
    print(f'  -> Macro - Precisão: {precisao_macro:.4f} | Recall: {recall_macro:.4f} | F1-Score: {f1_macro:.4f}\n')

    if acuracia_val <= melhor_acuracia_valid:
        caunt_paciencia += 1
        if caunt_paciencia >= paciencia:
            print(f"Early stopping na época {epoca+1} (sem melhora por {paciencia} épocas)")
            break
        continue

    caunt_paciencia = 1
    melhor_acuracia_valid = acuracia_val
    Path('model/best').mkdir(parents=True, exist_ok=True)
    torch.save(modelo.state_dict(), 'model/best/melhor_modelo.pth')

print("Treinamento Concluído!")
modelo.load_state_dict(torch.load('model/best/melhor_modelo.pth'))

In [ ]:
from pathlib import Path
import numpy as np
modelo.eval()
todas_previsoes = []
todos_rotulos = []

print("A avaliar o conjunto de teste...")
with torch.no_grad():
    for imagens, rotulos in loader_teste:
        imagens, rotulos = imagens.to(device), rotulos.to(device)
        saidas = modelo(imagens)
        previsoes = (torch.sigmoid(saidas) >= 0.5).float()
        todas_previsoes.extend(previsoes.cpu().numpy())
        todos_rotulos.extend(rotulos.cpu().numpy())

print("\n--- Relatório de Classificação (Teste) ---")
relatorio_texto = classification_report(todos_rotulos, todas_previsoes, target_names=['Saudável (0)', 'AVC (1)'])
print(relatorio_texto)

acc_final = accuracy_score(todos_rotulos, todas_previsoes) * 100
prec_final = precision_score(todos_rotulos, todas_previsoes, average='macro', zero_division=0) * 100
rec_final = recall_score(todos_rotulos, todas_previsoes, average='macro', zero_division=0) * 100
f1_final = f1_score(todos_rotulos, todas_previsoes, average='macro', zero_division=0) * 100

Path("model/best").mkdir(parents=True, exist_ok=True)
with open("model/best/relatorio_desempenho_avc.txt", "w", encoding="utf-8") as f:
    f.write("=========================================================\n")
    f.write("    RELATÓRIO TÉCNICO-CIENTÍFICO: DETECÇÃO DE AVC VIA CNN \n")
    f.write("=========================================================\n\n")
    f.write(f"Acurácia Geral no Teste: {acc_final:.2f}%\n")
    f.write(f"Precisão (Macro):       {prec_final/100:.4f}\n")
    f.write(f"Recall (Macro):         {rec_final/100:.4f}\n")
    f.write(f"F1-Score (Macro):       {f1_final/100:.4f}\n\n")
    f.write("--- Detalhamento por Classe ---\n")
    f.write(relatorio_texto)

fig1, eixos = plt.subplots(2, 2, figsize=(14, 10))
epocas_reais = len(historico_perda_treino)
eixo_x = range(1, epocas_reais + 1)

eixos[0, 0].plot(eixo_x, historico_perda_treino, color='#1070A0', linewidth=2, marker='o', markersize=4)
eixos[0, 0].set_title('1. Loss de Treino', fontsize=12, fontweight='bold', pad=10)
eixos[0, 0].set_xlabel('Épocas')
eixos[0, 0].set_ylabel('Loss')
eixos[0, 0].grid(True, linestyle='--', alpha=0.5)

eixos[0, 1].plot(eixo_x, historico_perda_validacao, color='#E66000', linewidth=2, marker='s', markersize=4)
eixos[0, 1].set_title('2. Loss de Validação', fontsize=12, fontweight='bold', pad=10)
eixos[0, 1].set_xlabel('Épocas')
eixos[0, 1].set_ylabel('Loss')
eixos[0, 1].grid(True, linestyle='--', alpha=0.5)

eixos[1, 0].plot(eixo_x, historico_acuracia_treino, color='#2CA02C', linewidth=2, marker='^', markersize=4)
eixos[1, 0].set_title('3. Acurácia de Treino (%)', fontsize=12, fontweight='bold', pad=10)
eixos[1, 0].set_xlabel('Épocas')
eixos[1, 0].set_ylabel('Acurácia (%)')
eixos[1, 0].grid(True, linestyle='--', alpha=0.5)

eixos[1, 1].plot(eixo_x, historico_acuracia_validacao, color='#9467BD', linewidth=2, marker='v', markersize=4)
eixos[1, 1].set_title('4. Acurácia de Validação (%)', fontsize=12, fontweight='bold', pad=10)
eixos[1, 1].set_xlabel('Épocas')
eixos[1, 1].set_ylabel('Acurácia (%)')
eixos[1, 1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle("CURVAS DE APRENDIZADO SEPARADAS", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

fig2, ax_cm = plt.subplots(figsize=(6, 5))

cm = confusion_matrix(todos_rotulos, todas_previsoes)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
labels_quadrantes = []
for i in range(2):
    for j in range(2):
        labels_quadrantes.append(f"{cm[i, j]}\n({cm_percent[i, j]*100:.1f}%)")
labels_quadrantes = np.asarray(labels_quadrantes).reshape(2,2)

sns.heatmap(cm, annot=labels_quadrantes, fmt='', cmap='Blues',
            xticklabels=['Saudável', 'AVC'], yticklabels=['Saudável', 'AVC'],
            cbar=False, square=True, annot_kws={"size": 12, "weight": "bold"}, ax=ax_cm)

ax_cm.set_title('Matriz de Confusão (Teste)', fontsize=13, fontweight='bold', pad=15)
ax_cm.set_ylabel('Classe Real (Gold Standard)', fontsize=11, fontweight='bold')
ax_cm.set_xlabel('Classe Prevista pelo Modelo', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()